# Experiment 0 - reproducibility
**Goal:** Confirm correct capturing of all sources of randomness and get a baseline for future experiments.

In [1]:
import torch
import pandas as pd
import numpy as np
import os

from src.utils import load_config, set_seed, build_input
from src.dataset import RSPositionDataset
from src.model import PositionPredictor, FocalLoss
from torch.utils.data import random_split
from src.train import train_model
from src.codec import HybridDecoder,  encode
from src.evaluate import evaluate_decoder, compute_metrics
from src.channel import qsc_erasure_channel

device = "cuda" if torch.cuda.is_available() else "cpu"

config = load_config("../configs/default.yaml")

## Dataset generation

In [2]:
set_seed(config["seed"])

n_samples = config["data"]["train_size"] + config["data"]["val_size"] + config["data"]["test_size"]
dataset = RSPositionDataset(
    size=n_samples,
    p_err=config["channel"]["symbol_error_prob"],
    p_erase=config["channel"]["symbol_erase_prob"]
)

train_dataset, temp_dataset = random_split(
    dataset, 
    [config["data"]["train_size"], config["data"]["val_size"] + config["data"]["test_size"]]
)

val_dataset, test_dataset = random_split(
    temp_dataset,
    [config["data"]["val_size"], config["data"]["test_size"]]
)

## Neural network training

In [ ]:
seeds = [15, 42, 57, 87, 114]
models = []
histories = []

criterion = FocalLoss(
    alpha=config["training"]["focal_alpha"],
    gamma=config["training"]["focal_gamma"]
)

for seed in seeds:
    set_seed(seed)
    model = PositionPredictor(
        hidden_size=config["model"]["hidden_size"],
        dropout=config["model"]["dropout"]
    )

    history = train_model(
        model=model,
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        criterion=criterion,
        epochs=config["training"]["epochs"],
        batch_size=config["training"]["batch_size"],
        lr=config["training"]["learning_rate"],
        verbose=True
    )

    models.append(model)
    histories.append(history)

## Upload training history

In [ ]:
all_rows = []

for i, seed in enumerate(seeds):
    for epoch, (tl, vl) in enumerate(zip(histories[i]["train_loss"], histories[i]["val_loss"])):
        all_rows.append({"seed": seed, "epoch": epoch, "train_loss": tl, "val_loss": vl})

hist_df = pd.DataFrame(all_rows)
hist_df.to_csv("../results/tables/exp00_train_history.csv", index=False)

## Key metrics evaluation

In [ ]:
rows = []

channel_fn = lambda codeword: qsc_erasure_channel(
    codeword, 
    p_err=config["channel"]["symbol_error_prob"],
    p_erase=config["channel"]["symbol_erase_prob"]
)

for i, seed in enumerate(seeds):
    decoder = HybridDecoder(models[i], threshold=config["training"]["threshold"], device=device)

    # FER, BER
    metrics = evaluate_decoder(decoder, channel_fn, num_samples=1000)
    metrics["seed"] = seed

    #Precision, Recall
    precisions = []
    recalls = []

    for _ in range(1000):
        msg = os.urandom(config["code"]["k"])
        codeword = encode(msg)
        noisy, erase_pos, error_pos = channel_fn(codeword)

        pred_pos = decoder.predict_positions(build_input(noisy))
        true_pos = [j for j in range(config["code"]["n"]) if noisy[j] != codeword[j]]

        pr = compute_metrics(pred_pos, true_pos)
        precisions.append(pr["precision"])
        recalls.append(pr["recall"])

    metrics["precision"] = np.mean(precisions)
    metrics["recall"] = np.mean(recalls)

    rows.append(metrics)

## Upload key metrics

In [ ]:
metrics_df = pd.DataFrame(rows)
metrics_df.to_csv("../results/tables/exp00_metrics.csv", index=False)